# Step 3A — Materialize all-cell ResolVI-corrected expression as AnnData Zarr

Run this notebook in the **same scvi-tools / ResolVI environment** used for
Step 2.

For every successfully trained sample it creates one authoritative all-cell
Zarr store:

```python
adata.X
# sparse integer Proseg counts

adata.layers["resolvi_corrected_10k"]
# dense float32 ResolVI posterior expected true expression,
# scaled to a common library size of 10,000
```

The dense corrected layer is decoded in cell chunks into a disk-backed NumPy
memmap and then written to chunked Zarr. The full corrected matrix is never
assembled in RAM. The raw matrix remains in `X`, so count-based analyses cannot
accidentally use corrected floating-point expression.

This notebook does **not** run pyUCell or reference annotation. Those are kept
separate so the corrected matrix is generated once and reused consistently.

## NumPy memmap → AnnData Zarr compatibility

The corrected matrix is decoded into a disk-backed `numpy.memmap` so it never
needs to be held completely in RAM. AnnData's I/O registry does not register
the `numpy.memmap` subclass directly. Before `write_zarr()`, this notebook now
wraps the same buffer with `np.asarray(memmap)`, producing an exact
`numpy.ndarray` view **without copying the full matrix into memory**.

Completed staging memmaps from an earlier failed Zarr write are detected and
reused. ResolVI decoding is skipped when the progress file confirms that every
row is already present.


In [1]:
# ---------------------------------------------------------------------
# Environment — run before importing torch, scvi-tools, or ResolVI
# ---------------------------------------------------------------------
import os

GPU_ID = "0"  # use a different physical GPU in each parallel kernel
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["PYTHONHASHSEED"] = "0"
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"
os.environ["OPENBLAS_NUM_THREADS"] = "16"

print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


CUDA_VISIBLE_DEVICES: 0


In [2]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import shutil
import time
import warnings
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import pyro
import scipy.sparse as sp
import torch
import zarr

import scvi
from scvi.external import RESOLVI

SAMPLE_INFO = {
    "Screen_39_21": {"patient": "patient_39_21", "cancer_type": "NSCLC", "biopsy_stage": "Screen"},
    "C2D15_39_21": {"patient": "patient_39_21", "cancer_type": "NSCLC", "biopsy_stage": "C2D15"},
    "Screen_17_26": {"patient": "patient_17_26", "cancer_type": "NSCLC", "biopsy_stage": "Screen"},
    "C2D15_17_26": {"patient": "patient_17_26", "cancer_type": "NSCLC", "biopsy_stage": "C2D15"},
    "Screen_18_23": {"patient": "patient_18_23", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_18_23": {"patient": "patient_18_23", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_16_22": {"patient": "patient_16_22", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_16_22": {"patient": "patient_16_22", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_30_16": {"patient": "patient_30_16", "cancer_type": "melanoma", "biopsy_stage": "Screen"},
    "C2D15_30_16": {"patient": "patient_30_16", "cancer_type": "melanoma", "biopsy_stage": "C2D15"},
    "Screen_23_25": {"patient": "patient_23_25", "cancer_type": "colon_cancer", "biopsy_stage": "Screen"},
    "C2D15_23_25": {"patient": "patient_23_25", "cancer_type": "colon_cancer", "biopsy_stage": "C2D15"},
}

print("scvi-tools:", scvi.__version__)
print("torch:", str(torch.__version__))
print("CUDA available:", torch.cuda.is_available())
print("Visible CUDA devices:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required for ResolVI decoding.")

torch.set_float32_matmul_precision("high")
scvi.settings.seed = 0


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Seed set to 0


scvi-tools: 1.5.0.post1
torch: 2.11.0+cu130
CUDA available: True
Visible CUDA devices: 1


In [3]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path("/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057")
PIPELINE_ROOT = PROJECT_ROOT / "tmp" / "proseg_resolvi_immune_enrichment_v1"
RESOLVI_ROOT = PIPELINE_ROOT / "02_resolvi"
ALLCELL_ZARR_ROOT = PIPELINE_ROOT / "03a_resolvi_allcell_zarr"
STAGING_ROOT = PIPELINE_ROOT / "03a_resolvi_allcell_zarr_staging"
ALLCELL_ZARR_ROOT.mkdir(parents=True, exist_ok=True)
STAGING_ROOT.mkdir(parents=True, exist_ok=True)

INCLUDE_COLON = True
SECTION_NAMES = [
    sample
    for sample, meta in SAMPLE_INFO.items()
    if INCLUDE_COLON or meta["cancer_type"] != "colon_cancer"
]

# For a quick integration test, set SECTION_NAMES to one completed sample.
# SECTION_NAMES = ["C2D15_23_25"]

SCVI_ACCELERATOR = "gpu"
SCVI_DEVICE_SPEC = 1  # one GPU visible through CUDA_VISIBLE_DEVICES
NORMALIZED_LIBRARY_SIZE = 10_000.0
DECODE_CELL_CHUNK = 1_000
POSTERIOR_BATCH_SIZE = 512
CORRECTED_DTYPE = np.float32
CORRECTED_LAYER = "resolvi_corrected_10k"

# Chunk shape used by AnnData.write_zarr for dense arrays.
ZARR_CHUNKS = (512, 2_048)

REUSE_COMPLETED_ZARR = True
OVERWRITE_ZARR = False
RESUME_MEMMAP = True
DELETE_STAGING_MEMMAP_AFTER_SUCCESS = True
CONTINUE_ON_ERROR = True

# The temporary memmap and the final Zarr coexist during export.
# Require a conservative amount of free space before decoding.
MIN_DISK_MULTIPLIER_OVER_DENSE = 2.2
MIN_EXTRA_FREE_GIB = 10.0

PIPELINE_VERSION = "2026-07-30-step3a-memmap-zarr-v2"

print("Samples:", SECTION_NAMES)
print("All-cell Zarr root:", ALLCELL_ZARR_ROOT)


Samples: ['Screen_39_21', 'C2D15_39_21', 'Screen_17_26', 'C2D15_17_26', 'Screen_18_23', 'C2D15_18_23', 'Screen_16_22', 'C2D15_16_22', 'Screen_30_16', 'C2D15_30_16', 'Screen_23_25', 'C2D15_23_25']
All-cell Zarr root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03a_resolvi_allcell_zarr


In [4]:
# ---------------------------------------------------------------------
# Paths and validation helpers
# ---------------------------------------------------------------------
def paths_for_sample(sample: str) -> dict[str, Path]:
    resolvi_dir = RESOLVI_ROOT / sample
    output_dir = ALLCELL_ZARR_ROOT / sample
    staging_dir = STAGING_ROOT / sample
    output_dir.mkdir(parents=True, exist_ok=True)
    staging_dir.mkdir(parents=True, exist_ok=True)
    return {
        "sample": Path(sample),
        "prepared": resolvi_dir / f"{sample}_resolvi_prepared.h5ad",
        "final_h5ad": resolvi_dir / f"{sample}_resolvi_annotated.h5ad",
        "model": resolvi_dir / "model",
        "zarr": output_dir / f"{sample}_resolvi_allcells.zarr",
        "summary": output_dir / f"{sample}_resolvi_allcells_zarr_summary.json",
        "memmap": staging_dir / f"{sample}_{CORRECTED_LAYER}.float32.dat",
        "progress": staging_dir / f"{sample}_{CORRECTED_LAYER}_progress.json",
    }


def write_json(payload, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temp.replace(path)


def names_hash(names) -> str:
    digest = hashlib.sha256()
    for value in names.astype(str):
        digest.update(value.encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def complete_model_checkpoint_exists(model_dir: Path) -> bool:
    return model_dir.is_dir() and (model_dir / "model.pt").is_file()


def exact_dense_bytes(n_obs: int, n_vars: int, dtype=np.float32) -> int:
    return int(n_obs) * int(n_vars) * np.dtype(dtype).itemsize


def available_disk_bytes(path: Path) -> int:
    return shutil.disk_usage(path).free


def close_memmap(array) -> None:
    try:
        array.flush()
    except Exception:
        pass
    mmap_obj = getattr(array, "_mmap", None)
    if mmap_obj is not None:
        try:
            mmap_obj.close()
        except Exception:
            pass


def validate_alignment(prepared: ad.AnnData, final: ad.AnnData) -> None:
    if prepared.shape != final.shape:
        raise ValueError(
            f"Prepared/final shape mismatch: {prepared.shape} versus {final.shape}."
        )
    if not prepared.obs_names.equals(final.obs_names):
        raise ValueError("Prepared and final obs_names are not identically ordered.")
    if not prepared.var_names.equals(final.var_names):
        raise ValueError("Prepared and final var_names are not identically ordered.")
    if not sp.issparse(final.X):
        final.X = sp.csr_matrix(final.X)
    final.X = sp.csr_matrix(final.X).astype(np.int32)


def validate_existing_zarr(paths: dict[str, Path], expected_shape) -> bool:
    if not paths["zarr"].exists() or not paths["summary"].exists():
        return False
    try:
        root = zarr.open_group(str(paths["zarr"]), mode="r")
        layer = root["layers"][CORRECTED_LAYER]
        if tuple(layer.shape) != tuple(expected_shape):
            return False
        summary = json.loads(paths["summary"].read_text(encoding="utf-8"))
        return (
            summary.get("pipeline_version") == PIPELINE_VERSION
            and summary.get("completed") is True
            and tuple(summary.get("shape", [])) == tuple(expected_shape)
        )
    except Exception:
        return False

def read_compatible_staging_progress(
    paths: dict[str, Path],
    shape: tuple[int, int],
) -> dict | None:
    """Return compatible staging progress, otherwise None.

    Compatibility intentionally does not require an identical pipeline version,
    because the v1 decoding output is numerically reusable; only the failed
    AnnData/Zarr handoff changed.
    """
    if not paths["progress"].exists() or not paths["memmap"].exists():
        return None

    try:
        payload = json.loads(
            paths["progress"].read_text(encoding="utf-8")
        )
    except Exception:
        return None

    expected_bytes = exact_dense_bytes(
        *shape,
        dtype=CORRECTED_DTYPE,
    )
    compatible = (
        tuple(payload.get("shape", [])) == tuple(shape)
        and payload.get("dtype") == np.dtype(CORRECTED_DTYPE).name
        and float(payload.get("library_size"))
        == float(NORMALIZED_LIBRARY_SIZE)
        and paths["memmap"].stat().st_size == expected_bytes
    )
    return payload if compatible else None


def completed_staging_memmap_exists(
    paths: dict[str, Path],
    shape: tuple[int, int],
) -> bool:
    """Check whether every corrected-expression row was already decoded."""
    payload = read_compatible_staging_progress(paths, shape)
    if payload is None:
        return False

    return (
        bool(payload.get("completed", False))
        and int(payload.get("next_row", 0)) == int(shape[0])
        and int(payload.get("total_entries_seen", 0))
        == int(shape[0]) * int(shape[1])
    )


def open_staging_memmap(
    paths: dict[str, Path],
    shape: tuple[int, int],
    mode: str = "r",
) -> np.memmap:
    """Open an existing compatible corrected-expression memmap."""
    payload = read_compatible_staging_progress(paths, shape)
    if payload is None:
        raise FileNotFoundError(
            "No compatible corrected-expression staging memmap was found."
        )

    return np.memmap(
        paths["memmap"],
        mode=mode,
        dtype=CORRECTED_DTYPE,
        shape=shape,
    )


def memmap_as_registered_ndarray(memmap: np.memmap) -> np.ndarray:
    """Return an exact ndarray view sharing the memmap buffer, with no copy.

    AnnData's I/O registry supports numpy.ndarray but not the numpy.memmap
    subclass. np.asarray(memmap) strips only the subclass wrapper.
    """
    array_view = np.asarray(memmap)

    if type(array_view) is not np.ndarray:
        raise TypeError(
            "Expected an exact numpy.ndarray view, observed "
            f"{type(array_view)!r}."
        )
    if array_view.shape != memmap.shape:
        raise ValueError(
            f"ndarray view shape {array_view.shape} does not match "
            f"memmap shape {memmap.shape}."
        )
    if array_view.dtype != memmap.dtype:
        raise TypeError(
            f"ndarray view dtype {array_view.dtype} does not match "
            f"memmap dtype {memmap.dtype}."
        )
    if not np.shares_memory(array_view, memmap):
        raise MemoryError(
            "The ndarray conversion unexpectedly copied the corrected matrix."
        )

    return array_view


def preflight_memmap_annData_zarr_write() -> None:
    """Verify the exact compatibility path on a tiny temporary object."""
    test_root = STAGING_ROOT / "_memmap_zarr_preflight"
    test_memmap_path = test_root / "test.float32.dat"
    test_zarr_path = test_root / "test.zarr"

    if test_root.exists():
        shutil.rmtree(test_root)
    test_root.mkdir(parents=True, exist_ok=True)

    test_memmap = None
    try:
        test_memmap = np.memmap(
            test_memmap_path,
            mode="w+",
            dtype=np.float32,
            shape=(4, 5),
        )
        test_memmap[:] = np.arange(20, dtype=np.float32).reshape(4, 5)
        test_memmap.flush()

        test_view = memmap_as_registered_ndarray(test_memmap)
        test_adata = ad.AnnData(
            X=sp.csr_matrix(np.eye(4, 5, dtype=np.int32)),
            obs=pd.DataFrame(index=[f"cell_{i}" for i in range(4)]),
            var=pd.DataFrame(index=[f"gene_{i}" for i in range(5)]),
        )
        test_adata.layers["corrected"] = test_view
        test_adata.write_zarr(str(test_zarr_path), chunks=(2, 3))

        test_root_group = zarr.open_group(
            str(test_zarr_path),
            mode="r",
        )
        written = np.asarray(
            test_root_group["layers"]["corrected"][:]
        )
        expected = np.arange(
            20,
            dtype=np.float32,
        ).reshape(4, 5)

        if not np.array_equal(written, expected):
            raise AssertionError(
                "The memmap-backed ndarray Zarr preflight changed values."
            )

        del test_adata.layers["corrected"]
        del test_view, test_adata, written
        print(
            "Memmap → ndarray-view → AnnData Zarr preflight passed."
        )
    finally:
        if test_memmap is not None:
            close_memmap(test_memmap)
        shutil.rmtree(test_root, ignore_errors=True)


In [5]:
# ---------------------------------------------------------------------
# Chunked decoding and Zarr export
# ---------------------------------------------------------------------
def decode_corrected_to_memmap(
    model,
    prepared: ad.AnnData,
    paths: dict[str, Path],
) -> tuple[np.memmap, dict]:
    shape = prepared.shape
    expected_bytes = exact_dense_bytes(*shape, dtype=CORRECTED_DTYPE)
    progress_payload = None

    if RESUME_MEMMAP:
        progress_payload = read_compatible_staging_progress(
            paths,
            shape,
        )
        if (
            progress_payload is None
            and (paths["progress"].exists() or paths["memmap"].exists())
        ):
            print("Discarding incompatible staging memmap.")
            paths["memmap"].unlink(missing_ok=True)
            paths["progress"].unlink(missing_ok=True)

    if progress_payload is None:
        memmap = np.memmap(
            paths["memmap"],
            mode="w+",
            dtype=CORRECTED_DTYPE,
            shape=shape,
        )
        next_row = 0
        nonzero_entries = 0
        total_entries_seen = 0
        min_value = None
        max_value = None
        progress_payload = {
            "pipeline_version": PIPELINE_VERSION,
            "shape": list(map(int, shape)),
            "dtype": np.dtype(CORRECTED_DTYPE).name,
            "library_size": float(NORMALIZED_LIBRARY_SIZE),
            "next_row": 0,
            "nonzero_entries": 0,
            "total_entries_seen": 0,
            "min_value": None,
            "max_value": None,
            "completed": False,
        }
        write_json(progress_payload, paths["progress"])
    else:
        memmap = np.memmap(
            paths["memmap"],
            mode="r+",
            dtype=CORRECTED_DTYPE,
            shape=shape,
        )
        next_row = int(progress_payload.get("next_row", 0))
        nonzero_entries = int(progress_payload.get("nonzero_entries", 0))
        total_entries_seen = int(progress_payload.get("total_entries_seen", 0))
        min_value = progress_payload.get("min_value")
        max_value = progress_payload.get("max_value")
        print(f"Resuming corrected-expression decoding at row {next_row:,}.")

    for start in range(next_row, prepared.n_obs, int(DECODE_CELL_CHUNK)):
        end = min(start + int(DECODE_CELL_CHUNK), prepared.n_obs)
        indices = np.arange(start, end, dtype=np.int64)

        try:
            chunk = model.get_normalized_expression(
                adata=prepared,
                indices=indices,
                gene_list=None,
                library_size=float(NORMALIZED_LIBRARY_SIZE),
                n_samples=1,
                return_mean=True,
                return_numpy=True,
                batch_size=int(POSTERIOR_BATCH_SIZE),
            )
        except RuntimeError as exc:
            if "out of memory" not in str(exc).lower():
                raise
            warnings.warn(
                "CUDA OOM during corrected decoding; retrying this chunk with "
                "posterior batch size 128."
            )
            torch.cuda.empty_cache()
            chunk = model.get_normalized_expression(
                adata=prepared,
                indices=indices,
                gene_list=None,
                library_size=float(NORMALIZED_LIBRARY_SIZE),
                n_samples=1,
                return_mean=True,
                return_numpy=True,
                batch_size=128,
            )

        chunk = np.asarray(chunk, dtype=CORRECTED_DTYPE)
        expected_chunk_shape = (end - start, prepared.n_vars)
        if chunk.shape != expected_chunk_shape:
            raise ValueError(
                f"Corrected chunk shape {chunk.shape}; expected {expected_chunk_shape}."
            )
        if not np.isfinite(chunk).all():
            raise FloatingPointError("Corrected expression contains non-finite values.")

        memmap[start:end, :] = chunk
        memmap.flush()

        nonzero_entries += int(np.count_nonzero(chunk))
        total_entries_seen += int(chunk.size)
        chunk_min = float(chunk.min())
        chunk_max = float(chunk.max())
        min_value = chunk_min if min_value is None else min(float(min_value), chunk_min)
        max_value = chunk_max if max_value is None else max(float(max_value), chunk_max)

        progress_payload.update(
            {
                "next_row": int(end),
                "nonzero_entries": int(nonzero_entries),
                "total_entries_seen": int(total_entries_seen),
                "min_value": float(min_value),
                "max_value": float(max_value),
                "completed": bool(end == prepared.n_obs),
            }
        )
        write_json(progress_payload, paths["progress"])
        print(
            f"Decoded {end:,}/{prepared.n_obs:,} cells "
            f"({100.0 * end / prepared.n_obs:.1f}%)."
        )
        del chunk
        gc.collect()
        torch.cuda.empty_cache()

    return memmap, progress_payload


def export_sample_allcell_zarr(sample: str) -> dict:
    paths = paths_for_sample(sample)
    print("\n" + "=" * 90)
    print("All-cell corrected Zarr sample:", sample)

    for key in ("prepared", "final_h5ad"):
        if not paths[key].exists():
            raise FileNotFoundError(paths[key])
    if not complete_model_checkpoint_exists(paths["model"]):
        raise FileNotFoundError(paths["model"] / "model.pt")

    final = ad.read_h5ad(paths["final_h5ad"])
    prepared = ad.read_h5ad(paths["prepared"])
    validate_alignment(prepared, final)

    if REUSE_COMPLETED_ZARR and not OVERWRITE_ZARR and validate_existing_zarr(
        paths, final.shape
    ):
        print("Reusing existing completed all-cell Zarr:", paths["zarr"])
        return json.loads(paths["summary"].read_text(encoding="utf-8"))

    dense_bytes = exact_dense_bytes(*final.shape, dtype=CORRECTED_DTYPE)
    final_h5ad_bytes = paths["final_h5ad"].stat().st_size
    required = int(
        dense_bytes * float(MIN_DISK_MULTIPLIER_OVER_DENSE)
        + final_h5ad_bytes
        + float(MIN_EXTRA_FREE_GIB) * 2**30
    )
    free = available_disk_bytes(ALLCELL_ZARR_ROOT)
    print(f"Corrected dense matrix estimate: {dense_bytes / 2**30:.1f} GiB")
    print(f"Available disk: {free / 2**30:.1f} GiB")
    print(f"Conservative required free disk: {required / 2**30:.1f} GiB")
    if free < required:
        raise OSError(
            "Insufficient free disk for the staging memmap plus final Zarr. "
            f"Required about {required / 2**30:.1f} GiB, available {free / 2**30:.1f} GiB."
        )

    pyro.clear_param_store()
    started = time.time()
    model = None

    if completed_staging_memmap_exists(paths, prepared.shape):
        print(
            "Reusing completed corrected-expression staging memmap; "
            "ResolVI decoding will not be repeated."
        )
        progress = read_compatible_staging_progress(
            paths,
            prepared.shape,
        )
        memmap = open_staging_memmap(
            paths,
            prepared.shape,
            mode="r",
        )
    else:
        model = RESOLVI.load(
            str(paths["model"]),
            adata=prepared,
            accelerator=SCVI_ACCELERATOR,
            device=SCVI_DEVICE_SPEC,
        )
        memmap, progress = decode_corrected_to_memmap(
            model,
            prepared,
            paths,
        )

    memmap.flush()
    corrected_array_view = memmap_as_registered_ndarray(memmap)
    print(
        "Corrected layer handoff:",
        {
            "source_type": type(memmap).__name__,
            "writer_type": type(corrected_array_view).__name__,
            "shares_memory": bool(
                np.shares_memory(corrected_array_view, memmap)
            ),
            "shape": tuple(map(int, corrected_array_view.shape)),
            "dtype": str(corrected_array_view.dtype),
        },
    )

    final.layers[CORRECTED_LAYER] = corrected_array_view
    final.uns["resolvi_corrected_expression"] = {
        "pipeline_version": PIPELINE_VERSION,
        "layer": CORRECTED_LAYER,
        "raw_count_location": "X",
        "library_size": float(NORMALIZED_LIBRARY_SIZE),
        "dtype": np.dtype(CORRECTED_DTYPE).name,
        "interpretation": (
            "ResolVI posterior expected true normalized expression; "
            "not observed integer counts"
        ),
        "decode_cell_chunk": int(DECODE_CELL_CHUNK),
        "posterior_batch_size": int(POSTERIOR_BATCH_SIZE),
    }

    temp_zarr = paths["zarr"].with_name(paths["zarr"].name + ".tmp")
    if temp_zarr.exists():
        shutil.rmtree(temp_zarr)
    if paths["zarr"].exists():
        if not OVERWRITE_ZARR:
            raise FileExistsError(paths["zarr"])
        shutil.rmtree(paths["zarr"])

    print("Writing AnnData Zarr ...")
    final.write_zarr(str(temp_zarr), chunks=ZARR_CHUNKS)
    shutil.move(str(temp_zarr), str(paths["zarr"]))

    root = zarr.open_group(str(paths["zarr"]), mode="r")
    corrected_disk = root["layers"][CORRECTED_LAYER]
    if tuple(corrected_disk.shape) != tuple(final.shape):
        raise ValueError("Written corrected Zarr layer has the wrong shape.")

    obs_hash = names_hash(final.obs_names)
    var_hash = names_hash(final.var_names)
    density = (
        float(progress["nonzero_entries"]) / float(progress["total_entries_seen"])
        if progress["total_entries_seen"]
        else float("nan")
    )
    summary = {
        "pipeline_version": PIPELINE_VERSION,
        "sample": sample,
        **SAMPLE_INFO[sample],
        "completed": True,
        "zarr_path": str(paths["zarr"]),
        "shape": list(map(int, final.shape)),
        "raw_count_location": "X",
        "corrected_layer": CORRECTED_LAYER,
        "corrected_dtype": str(corrected_disk.dtype),
        "corrected_chunks": list(map(int, corrected_disk.chunks)),
        "corrected_exact_density": density,
        "corrected_min": progress.get("min_value"),
        "corrected_max": progress.get("max_value"),
        "library_size": float(NORMALIZED_LIBRARY_SIZE),
        "obs_names_sha256": obs_hash,
        "var_names_sha256": var_hash,
        "runtime_minutes": (time.time() - started) / 60.0,
    }
    write_json(summary, paths["summary"])
    print("Saved:", paths["zarr"])
    print("Corrected exact density:", f"{density:.4%}")

    # Release the ndarray view before closing and deleting the memmap.
    del final.layers[CORRECTED_LAYER]
    del corrected_array_view
    close_memmap(memmap)

    if model is not None:
        del model
    del memmap, prepared, final
    pyro.clear_param_store()
    gc.collect()
    torch.cuda.empty_cache()

    if DELETE_STAGING_MEMMAP_AFTER_SUCCESS:
        paths["memmap"].unlink(missing_ok=True)
        paths["progress"].unlink(missing_ok=True)

    return summary


## Run selected samples

For four-way parallelization, open four kernels with distinct `GPU_ID` values
and disjoint `SECTION_NAMES`. Avoid writing the same sample from two kernels.


## Memmap/Zarr writer preflight

This writes and rereads a tiny temporary AnnData Zarr using the exact disk-backed ndarray-view path used for the full corrected matrices. It should complete in seconds and fails before any sample work if the installed AnnData/Zarr combination is incompatible.


In [6]:
preflight_memmap_annData_zarr_write()


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


Memmap → ndarray-view → AnnData Zarr preflight passed.


In [7]:
allcell_results = {}
allcell_failures = {}

for sample in SECTION_NAMES:
    try:
        allcell_results[sample] = export_sample_allcell_zarr(sample)
    except Exception as exc:
        allcell_failures[sample] = repr(exc)
        print(f"[FAILED] {sample}: {type(exc).__name__}: {exc}")
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        pyro.clear_param_store()
        gc.collect()
        torch.cuda.empty_cache()

pd.DataFrame.from_dict(allcell_results, orient="index").to_csv(
    ALLCELL_ZARR_ROOT / "all_samples_allcell_zarr_summary.csv"
)
write_json(
    allcell_failures,
    ALLCELL_ZARR_ROOT / "all_samples_allcell_zarr_failures.json",
)

print("Completed:", sorted(allcell_results))
print("Failures:", json.dumps(allcell_failures, indent=2))



All-cell corrected Zarr sample: Screen_39_21
Corrected dense matrix estimate: 1.5 GiB
Available disk: 8589409811.5 GiB
Conservative required free disk: 13.4 GiB
Reusing completed corrected-expression staging memmap; ResolVI decoding will not be repeated.
Corrected layer handoff: {'source_type': 'memmap', 'writer_type': 'ndarray', 'shares_memory': True, 'shape': (22950, 18003), 'dtype': 'float32'}
Writing AnnData Zarr ...


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03a_resolvi_allcell_zarr/Screen_39_21/Screen_39_21_resolvi_allcells.zarr
Corrected exact density: 100.0000%

All-cell corrected Zarr sample: C2D15_39_21
Corrected dense matrix estimate: 0.7 GiB
Available disk: 8589409811.5 GiB
Conservative required free disk: 11.6 GiB
Reusing completed corrected-expression staging memmap; ResolVI decoding will not be repeated.
Corrected layer handoff: {'source_type': 'memmap', 'writer_type': 'ndarray', 'shares_memory': True, 'shape': (10822, 17282), 'dtype': 'float32'}
Writing AnnData Zarr ...


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03a_resolvi_allcell_zarr/C2D15_39_21/C2D15_39_21_resolvi_allcells.zarr
Corrected exact density: 100.0000%

All-cell corrected Zarr sample: Screen_17_26
Corrected dense matrix estimate: 3.2 GiB
Available disk: 8589409811.5 GiB
Conservative required free disk: 17.3 GiB
Reusing completed corrected-expression staging memmap; ResolVI decoding will not be repeated.
Corrected layer handoff: {'source_type': 'memmap', 'writer_type': 'ndarray', 'shares_memory': True, 'shape': (47896, 18060), 'dtype': 'float32'}
Writing AnnData Zarr ...


...

Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03a_resolvi_allcell_zarr/C2D15_30_16/C2D15_30_16_resolvi_allcells.zarr
Corrected exact density: 99.9764%

All-cell corrected Zarr sample: Screen_23_25
Corrected dense matrix estimate: 1.6 GiB
Available disk: 8589409799.1 GiB
Conservative required free disk: 13.7 GiB
Reusing completed corrected-expression staging memmap; ResolVI decoding will not be repeated.
Corrected layer handoff: {'source_type': 'memmap', 'writer_type': 'ndarray', 'shares_memory': True, 'shape': (25776, 17171), 'dtype': 'float32'}
Writing AnnData Zarr ...


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03a_resolvi_allcell_zarr/Screen_23_25/Screen_23_25_resolvi_allcells.zarr
Corrected exact density: 100.0000%

All-cell corrected Zarr sample: C2D15_23_25
Corrected dense matrix estimate: 0.6 GiB
Available disk: 8589409799.1 GiB
Conservative required free disk: 11.3 GiB
Reusing completed corrected-expression staging memmap; ResolVI decoding will not be repeated.
Corrected layer handoff: {'source_type': 'memmap', 'writer_type': 'ndarray', 'shares_memory': True, 'shape': (10243, 15547), 'dtype': 'float32'}
Writing AnnData Zarr ...


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


Saved: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/03a_resolvi_allcell_zarr/C2D15_23_25/C2D15_23_25_resolvi_allcells.zarr
Corrected exact density: 100.0000%
Completed: ['C2D15_16_22', 'C2D15_17_26', 'C2D15_18_23', 'C2D15_23_25', 'C2D15_30_16', 'C2D15_39_21', 'Screen_16_22', 'Screen_17_26', 'Screen_18_23', 'Screen_23_25', 'Screen_30_16', 'Screen_39_21']
Failures: {}


In [8]:
#also record environment location
import sys
print(sys.executable)

/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python


## Lazy downstream access example

```python
import anndata as ad

lazy = ad.experimental.read_lazy(
    "/path/to/Screen_30_16_resolvi_allcells.zarr"
)

lazy.X
# lazy sparse raw counts

lazy.layers["resolvi_corrected_10k"]
# lazy dense corrected expression
```
